# Stiff van der Pol: when to use a stiff solver

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZetaOne/pyauto-xpp/blob/main/tutorials/notebooks/11_stiff_van_der_pol.ipynb)

The van der Pol relaxation oscillator with `mu = 1000`: the fast and slow time scales
differ by a factor of a million. An explicit method would need a tiny step; XPP's stiff
(Rosenbrock) solver takes steps of 20 time units.

*Based on an example from [XPP Studio](https://xpp-studio.github.io/xpp-studio/) (Peter Ashwin and Bard Ermentrout, GPL-3.0). Every number below is computed by XPPAUT 8.2 and its AUTO through pyauto-xpp.*

Running in Google Colab? This cell installs pyauto-xpp first; elsewhere it does nothing.

In [1]:
import sys
if "google.colab" in sys.modules:
    %pip -q install pyauto-xpp "plotly>=6" anywidget ipywidgets --find-links https://github.com/ZetaOne/pyauto-xpp/releases/expanded_assets/v0.4.0
    import urllib.request
    urllib.request.urlretrieve("https://raw.githubusercontent.com/ZetaOne/pyauto-xpp/main/tutorials/notebooks/xpptut.py", "xpptut.py")
    from google.colab import output
    output.enable_custom_widget_manager()

In [2]:
import plotly.graph_objects as go
from pyxpp.builder import ModelBuilder
from xpptut import Explorer, load, orbit_diagram, show_figure

## The model

Written with `ModelBuilder`, statement by statement as in an `.ode` file. Edit it and rerun the cells below.

In [3]:
mb = (ModelBuilder("stiff_van_der_pol")
      .ode("x", "mu*(x-x^3/3-y)")
      .ode("y", "x/mu")
      .parameter(mu=1000)
      .initial(x=2, y=0.6667)
      .option(meth="stiff", total=4000, dt=20, toler=1e-6))
print(mb.render())

x'=mu*(x-x^3/3-y)
y'=x/mu
par mu=1000
init x=2,y=0.6667
@ meth=stiff,total=4000,dt=20,toler=1e-06
done



## Explore

The sliders, the direction-field menu and the AUTO panel all call XPP. `ex.m` is the pyxpp model, for anything else.

In [4]:
ex = Explorer(mb, kind="plane", x="x", y="y", window=(-2.5, 2.5, -1, 1),
              sliders={"mu": (1, 2000)}, title="Stiff van der Pol")
ex

(run this cell with the kernel to get the interactive explorer)

## Going further

**Solvers compared**, with `Run` objects: the same model, different integrators.

In [5]:
import time
from pyxpp import Run
for r in (Run(meth="stiff", dt=20, tol=1e-6), Run(meth="cvode", dt=20, tol=1e-8, atol=1e-8), Run(meth="rk4", dt=0.001)):
    t0 = time.time(); tr = ex.m.run(r.replace(total=4000)); dt = time.time() - t0
    print(f"{r.meth:6s} rows {len(tr):8d}  {dt:6.2f} s   x at t=4000: {tr['x'][-1]:+.4f}")

stiff  rows      201    0.01 s   x at t=4000: -1.1955


cvode  rows      201    0.00 s   x at t=4000: -1.1955


rk4    rows  4000001    1.91 s   x at t=4000: -1.2393


## Things to try

- Lower `mu` to 1–5: the cycle rounds off and is no longer stiff.